# TASK-021 — test trên Kaggle trước khi đưa lên GitHub

**NOT RUN — pending Kaggle validation.** Mặc định dùng source ZIP upload từ máy.
Không cần commit/push, không cần EXPECTED_COMMIT khi SOURCE_MODE="upload".

1. Import notebook này vào Kaggle.
2. Thêm ZIP `task021-source-*.zip` vào dữ liệu đầu vào riêng tư của notebook.
   ZIP còn nguyên hoặc đã được Kaggle giải nén đều được hỗ trợ.
3. Giữ SOURCE_MODE="upload", SOURCE_PATH="" để tự tìm đúng một gói source.
4. Bật Internet để chuẩn bị toolchain, rồi Restart Session và Run All.

Nếu có nhiều gói, đặt SOURCE_PATH thành đường dẫn ZIP hoặc thư mục chứa
`task021-source-manifest.json` trong `/kaggle/input`.
Chế độ GitHub tùy chọn: SOURCE_MODE="github", đặt GIT_REF và EXPECTED_COMMIT.
Build/test/LLM chỉ chạy trên Kaggle; unit và HTTP mock không cần GPU.


In [ ]:
SOURCE_MODE = "upload"  # "upload" để test trước; "github" để clone sau.
SOURCE_PATH = ""  # Tự tìm một gói source trong /kaggle/input; có thể điền path cụ thể.
REPOSITORY = "https://github.com/vhnam1408/AI-Agent.git"
GIT_REF = "task021/nam-deadline-confirmation"
EXPECTED_COMMIT = ""  # Chỉ bắt buộc khi SOURCE_MODE="github"; upload để trống.
INSTALL_TOOLCHAIN = True  # Cài riêng trên Kaggle; dùng lại toolchain nếu đã có.
TOOLCHAIN = "/kaggle/working/task021-toolchain"
RUN_ROOT_REGRESSION = True
RUN_LIVE = False
LIVE_MODEL = ""  # Tên model text + vision + tools đã có trong Ollama Kaggle.
OLLAMA_URL = "http://127.0.0.1:11434"


In [ ]:
import hashlib
import json
import os
from pathlib import Path, PurePosixPath
import re
import shutil
import subprocess
import sys
import tarfile
import time
import urllib.request
import uuid
import xml.etree.ElementTree as ET
from zipfile import ZipFile, ZIP_DEFLATED

if not Path("/kaggle/working").is_dir() or not os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
    raise RuntimeError("Chỉ thực thi trong Kaggle Notebook; không chạy local/sandbox.")
if SOURCE_MODE not in ("upload", "github"):
    raise ValueError('SOURCE_MODE phải là "upload" hoặc "github".')
if SOURCE_MODE == "github":
    if not re.fullmatch(r"[0-9a-fA-F]{40}", EXPECTED_COMMIT):
        raise ValueError("Chế độ github cần EXPECTED_COMMIT đủ 40 ký tự; dùng upload nếu chưa push.")
    if not GIT_REF or GIT_REF.startswith("-"):
        raise ValueError("GIT_REF phải là branch/ref tường minh.")
WORK = Path("/kaggle/working") / ("task021-" + time.strftime("%Y%m%d-%H%M%S", time.gmtime()) + "-" + uuid.uuid4().hex[:8])
EVIDENCE = WORK / "evidence"
EVIDENCE.mkdir(parents=True)
REPO = WORK / "AI-Agent"
ZIP = WORK.with_suffix(".zip")
report = {"source_mode": SOURCE_MODE, "ref": GIT_REF if SOURCE_MODE == "github" else None,
          "expected_commit": EXPECTED_COMMIT if SOURCE_MODE == "github" else None, "commands": [],
          "statuses": {name: "NOT RUN" for name in
                       ("toolchain", "source", "build", "task17_unit", "task21_unit",
                        "task17_http_integration", "task17_live_criteria", "root_regression", "live")}}


def archive():
    # Chỉ đóng gói evidence được chọn; không chứa repo, token, model hoặc biến môi trường.
    required = ("toolchain", "source", "build", "task17_unit", "task21_unit",
                "task17_http_integration", "task17_live_criteria", "root_regression")
    if RUN_LIVE:
        required += ("live",)
    states = [report["statuses"][name] for name in required]
    report["overall_status"] = ("FAIL" if "FAIL" in states else "BLOCKED" if "BLOCKED" in states
                                else "PASS" if all(state == "PASS" for state in states) else "NOT RUN")
    (EVIDENCE / "report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    with ZipFile(ZIP, "w", ZIP_DEFLATED) as output:
        for path in sorted(EVIDENCE.rglob("*")):
            if path.is_file():
                output.write(path, path.relative_to(EVIDENCE))


def run_logged(name, command, cwd=None, check=True, timeout=1200):
    # Ghi cả stdout/stderr và exit code; thất bại vẫn xuất ZIP trước khi dừng.
    args = [str(part) for part in command]
    try:
        result = subprocess.run(args, cwd=cwd, text=True, capture_output=True, timeout=timeout)
    except Exception as error:
        for stream, attribute in (("stdout", "stdout"), ("stderr", "stderr")):
            partial = getattr(error, attribute, "") or ""
            if isinstance(partial, bytes):
                partial = partial.decode("utf-8", errors="replace")
            (EVIDENCE / (name + "." + stream + ".txt")).write_text(partial, encoding="utf-8")
        report["commands"].append({"name": name, "status": "FAIL", "error": str(error)})
        archive()
        raise
    (EVIDENCE / (name + ".stdout.txt")).write_text(result.stdout, encoding="utf-8")
    (EVIDENCE / (name + ".stderr.txt")).write_text(result.stderr, encoding="utf-8")
    report["commands"].append({"name": name, "returncode": result.returncode,
                               "status": "PASS" if result.returncode == 0 else "FAIL"})
    archive()
    if check and result.returncode:
        raise RuntimeError(f"{name}: exit {result.returncode}. Xem log trong {ZIP}")
    return result


def copy_test_artifacts(build, prefix):
    # Chép JSON unit/HTTP và CTest XML/log kể cả khi có test FAIL.
    destination = EVIDENCE / prefix
    destination.mkdir(exist_ok=True)
    for path in build.glob("*.json"):
        shutil.copy2(path, destination / path.name)
    cache_file = build / "CMakeCache.txt"
    if cache_file.is_file():
        shutil.copy2(cache_file, destination / cache_file.name)
    testing = build / "Testing"
    if testing.exists():
        for path in testing.rglob("*"):
            if path.is_file() and path.suffix in (".xml", ".log", ".txt"):
                target = destination / path.relative_to(build)
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(path, target)


def ctest_statuses(build):
    # Đọc kết quả CTest thực tế; test thiếu hoặc skipped không được xem là PASS.
    statuses = {}
    for xml in (build / "Testing").glob("*/Test.xml"):
        for test in ET.parse(xml).findall("./Testing/Test"):
            statuses[test.findtext("Name")] = "PASS" if test.get("Status") == "passed" else "FAIL"
    return statuses


archive()
print("Evidence:", ZIP)


## Compiler và dependency chỉ trong Kaggle

Dùng công thức micromamba của notebook TASK-013 hiện có, thêm SQLite cho regression
root. Nếu tắt `INSTALL_TOOLCHAIN`, chuẩn bị môi trường tương đương trước.
Cell kiểm thật khả năng C++23/`std::expected`, không chỉ dựa vào số phiên bản.


In [ ]:
try:
    toolchain = Path(TOOLCHAIN).resolve()
    # Tên executable phải là micromamba để shell hook và run kích hoạt đúng môi trường.
    mamba = Path("/kaggle/working/task021-mamba/bin/micromamba")
    mamba.parent.mkdir(parents=True, exist_ok=True)
    os.environ["MAMBA_EXE"] = str(mamba)
    os.environ["MAMBA_ROOT_PREFIX"] = "/kaggle/working/task021-mamba-root"
    os.environ["PATH"] = str(mamba.parent) + os.pathsep + os.environ.get("PATH", "")
    if INSTALL_TOOLCHAIN:
        if not mamba.exists():
            package = WORK / "micromamba.tar.bz2"
            urllib.request.urlretrieve("https://micro.mamba.pm/api/micromamba/linux-64/latest", package)
            report["micromamba_package_sha256"] = hashlib.sha256(package.read_bytes()).hexdigest()
            with tarfile.open(package, "r:bz2") as source:
                member = source.extractfile("bin/micromamba")
                if member is None:
                    raise RuntimeError("Thiếu bin/micromamba trong package")
                mamba.write_bytes(member.read())
            mamba.chmod(0o755)
        if not (toolchain / "conda-meta").is_dir():
            run_logged("toolchain_install", [mamba, "create", "-y", "-p", toolchain, "-c", "conda-forge",
                       "gxx_linux-64>=15", "cmake", "pkg-config", "libcurl", "sqlite", "libsqlite"], timeout=1800)
    CMAKE_OPTIONS = ["-DCMAKE_BUILD_TYPE=Release", "-DCMAKE_EXPORT_COMPILE_COMMANDS=ON"]
    if mamba.exists() and (toolchain / "conda-meta").is_dir():
        compilers = sorted((toolchain / "bin").glob("*conda-linux-gnu-c++"))
        if not compilers:
            raise RuntimeError("Toolchain không có compiler C++")
        CXX = str(compilers[0])
        CMAKE_BIN = str(toolchain / "bin/cmake")
        CTEST_BIN = str(toolchain / "bin/ctest")
        if not Path(CMAKE_BIN).is_file() or not Path(CTEST_BIN).is_file():
            raise RuntimeError("Toolchain thiếu cmake/ctest; cần cài đủ package trên Kaggle")
        PREFIX = [str(mamba), "run", "-p", str(toolchain)]
        # Kiểm activation thực sự, không chỉ tin exit code khi shell hook báo lỗi.
        activation = run_logged("toolchain_activation", PREFIX + [sys.executable, "-c",
            "import os,json; print(json.dumps({'prefix':os.environ.get('CONDA_PREFIX')}))"])
        if Path(json.loads(activation.stdout)["prefix"]).resolve() != toolchain:
            raise RuntimeError("micromamba chưa kích hoạt đúng CONDA_PREFIX")
        curl_library = toolchain / "lib/libcurl.so"
        sqlite_library = toolchain / "lib/libsqlite3.so"
        for dependency in (toolchain / "include/curl/curl.h", toolchain / "include/sqlite3.h",
                           curl_library, sqlite_library):
            if not dependency.is_file():
                raise RuntimeError("Toolchain thiếu dependency: " + str(dependency))
        # Giữ headers/libraries cùng prefix với compiler, tránh kéo /usr/include vào sysroot Conda.
        CMAKE_OPTIONS += ["-DCMAKE_PREFIX_PATH=" + str(toolchain),
                          "-DCMAKE_BUILD_RPATH=" + str(toolchain / "lib"),
                          "-DCURL_NO_CURL_CMAKE=ON",
                          "-DCURL_INCLUDE_DIR=" + str(toolchain / "include"),
                          "-DCURL_LIBRARY=" + str(curl_library),
                          "-DCURL_LIBRARY_RELEASE=" + str(curl_library),
                          "-DSQLite3_INCLUDE_DIR=" + str(toolchain / "include"),
                          "-DSQLite3_LIBRARY=" + str(sqlite_library)]
    else:
        CXX = os.environ.get("CXX") or shutil.which("g++")
        CMAKE_BIN = shutil.which("cmake")
        CTEST_BIN = shutil.which("ctest")
        PREFIX = []
    if not CXX or not CMAKE_BIN or not CTEST_BIN:
        raise RuntimeError("Thiếu compiler/cmake/ctest; cài toolchain trên Kaggle")
    CMAKE_OPTIONS += ["-DCMAKE_CXX_COMPILER=" + CXX]
    report["cmake_options"] = CMAKE_OPTIONS
    report["compiler"] = run_logged("compiler_version", PREFIX + [CXX, "--version"]).stdout
    report["cmake"] = run_logged("cmake_version", PREFIX + [CMAKE_BIN, "--version"]).stdout
    probe = WORK / "expected_probe.cpp"
    probe.write_text("#include <expected>\nint main(){std::expected<int,int> x=1; return x.value()-1;}\n", encoding="utf-8")
    run_logged("expected_probe_build", PREFIX + [CXX, "-std=c++23", str(probe), "-o", str(WORK / "expected_probe")])
    run_logged("expected_probe_run", PREFIX + [WORK / "expected_probe"])
    print_probe = WORK / "print_probe.cpp"
    print_probe.write_text("#include <print>\nint main(){std::print(\"{}\", 1); return 0;}\n", encoding="utf-8")
    run_logged("print_probe_build", PREFIX + [CXX, "-std=c++23", str(print_probe), "-lstdc++exp", "-o", str(WORK / "print_probe")])
    run_logged("print_probe_run", PREFIX + [WORK / "print_probe"])
    # Preflight dùng đúng CMake và các dependency của dự án, có chrono/time_t/curl/SQLite.
    dependency_source = WORK / "dependency-probe"
    dependency_source.mkdir()
    (dependency_source / "main.cpp").write_text(
        "#include <chrono>\n#include <expected>\n#include <print>\n#include <curl/curl.h>\n#include <sqlite3.h>\n"
        "int main(){std::expected<int,int> x=1; std::print(\"{}\", x.value()); "
        "auto t=std::chrono::system_clock::to_time_t(std::chrono::system_clock::now()); "
        "(void)t; if(curl_global_init(CURL_GLOBAL_DEFAULT)!=0) return 1; "
        "curl_global_cleanup(); return sqlite3_initialize();}\n", encoding="utf-8")
    (dependency_source / "CMakeLists.txt").write_text(
        "cmake_minimum_required(VERSION 3.20)\nproject(dependency_probe LANGUAGES CXX)\n"
        "set(CMAKE_CXX_STANDARD 23)\nset(CMAKE_CXX_STANDARD_REQUIRED ON)\n"
        "find_package(CURL REQUIRED)\nfind_package(SQLite3 REQUIRED)\n"
        "add_executable(dependency_probe main.cpp)\n"
        "target_link_libraries(dependency_probe PRIVATE CURL::libcurl SQLite::SQLite3)\n"
        "if(CMAKE_CXX_COMPILER_ID STREQUAL \"GNU\")\n"
        "target_link_libraries(dependency_probe PRIVATE stdc++exp)\nendif()\n", encoding="utf-8")
    dependency_build = WORK / "dependency-probe-build"
    run_logged("dependency_probe_configure", PREFIX + [CMAKE_BIN, "-S", dependency_source, "-B", dependency_build] + CMAKE_OPTIONS)
    # Lưu các cache quan trọng; không ghi toàn bộ biến môi trường hoặc secret.
    cache = {}
    for line in (dependency_build / "CMakeCache.txt").read_text(encoding="utf-8").splitlines():
        if not line.startswith(("#", "//")) and ":" in line and "=" in line:
            key = line.split(":", 1)[0]
            if key in ("CURL_INCLUDE_DIR", "CURL_LIBRARY", "SQLite3_INCLUDE_DIR", "SQLite3_LIBRARY"):
                cache[key] = line.split("=", 1)[1]
    if PREFIX:
        for key in ("CURL_INCLUDE_DIR", "CURL_LIBRARY", "SQLite3_INCLUDE_DIR", "SQLite3_LIBRARY"):
            if key not in cache or not Path(cache[key]).resolve().is_relative_to(toolchain):
                raise RuntimeError("Dependency bị lấy ngoài toolchain: " + key)
    report["resolved_dependencies"] = cache
    (EVIDENCE / "resolved_dependencies.json").write_text(json.dumps(cache, indent=2), encoding="utf-8")
    run_logged("dependency_probe_build", PREFIX + [CMAKE_BIN, "--build", dependency_build, "--parallel", "2", "--verbose"])
    run_logged("dependency_probe_run", PREFIX + [dependency_build / "dependency_probe"])
    report["statuses"]["toolchain"] = "PASS"
except Exception as error:
    report["statuses"]["toolchain"] = "BLOCKED"
    report["error"] = str(error)
    raise
finally:
    archive()


In [ ]:
def uploaded_source():
    # Tìm gói ZIP hoặc dataset đã giải nén; không chọn tùy ý khi có nhiều gói.
    if SOURCE_PATH:
        selected = Path(SOURCE_PATH)
    else:
        manifests = sorted(Path("/kaggle/input").rglob("task021-source-manifest.json"))
        candidates = [item.parent for item in manifests]
        if not candidates:
            candidates = sorted(Path("/kaggle/input").rglob("task021-source-*.zip"))
        if len(candidates) != 1:
            raise ValueError("Cần đúng một gói source upload; đặt SOURCE_PATH. Tìm thấy: " + str(candidates))
        selected = candidates[0]
    if not selected.exists():
        raise FileNotFoundError("Không tìm thấy SOURCE_PATH: " + str(selected))
    return selected


def copy_uploaded_source(selected):
    # Chỉ chép file có trong manifest, kiểm đường dẫn và hash; không extractall ZIP.
    zipped = selected.is_file()
    package = ZipFile(selected) if zipped else None
    try:
        raw = (package.read("task021-source-manifest.json") if zipped
               else (selected / "task021-source-manifest.json").read_bytes())
        snapshot = json.loads(raw)
        if snapshot.get("schema_version") != 1 or snapshot.get("kind") != "task021-working-tree":
            raise ValueError("Manifest không phải gói source TASK-021 được hỗ trợ")
        files = snapshot.get("files")
        if not isinstance(files, dict) or not files:
            raise ValueError("Manifest thiếu danh sách file")
        identity = hashlib.sha256(json.dumps(files, sort_keys=True, separators=(",", ":")).encode()).hexdigest()
        if identity != snapshot.get("source_id"):
            raise ValueError("source_id không khớp manifest")
        if zipped:
            names = [item.filename for item in package.infolist() if not item.is_dir()]
            expected = {"task021-source-manifest.json"} | {"AI-Agent/" + name for name in files}
            if len(names) != len(set(names)) or set(names) != expected:
                raise ValueError("ZIP có file trùng, thừa hoặc thiếu so với manifest")
        REPO.mkdir()
        for relative, expected_hash in files.items():
            name = PurePosixPath(relative)
            if (not relative or name.is_absolute() or name.as_posix() != relative or
                    any(part in ("..", ".git") for part in name.parts) or
                    "\\" in relative or ":" in relative):
                raise ValueError("Path source không hợp lệ: " + relative)
            if not isinstance(expected_hash, str) or not re.fullmatch(r"[0-9a-f]{64}", expected_hash):
                raise ValueError("Hash source không hợp lệ: " + relative)
            if zipped:
                content = package.read("AI-Agent/" + relative)
            else:
                source_file = selected / "AI-Agent" / relative
                if not source_file.resolve().is_relative_to((selected / "AI-Agent").resolve()):
                    raise ValueError("Source nằm ngoài thư mục upload")
                content = source_file.read_bytes()
            if hashlib.sha256(content).hexdigest() != expected_hash:
                raise ValueError("Hash source không khớp: " + relative)
            target = REPO / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(content)
        (EVIDENCE / "uploaded_source_manifest.json").write_bytes(raw)
        report["source_path"] = str(selected)
        report["source_id"] = identity
        report["base_commit"] = snapshot.get("base_commit")
        report["commit"] = None  # Đây là working tree, không phải source của base_commit.
        if zipped:
            report["source_zip_sha256"] = hashlib.sha256(selected.read_bytes()).hexdigest()
        return files
    finally:
        if package is not None:
            package.close()


try:
    if SOURCE_MODE == "upload":
        manifest = copy_uploaded_source(uploaded_source())
    else:
        run_logged("clone", ["git", "clone", "--branch", GIT_REF, "--single-branch", REPOSITORY, REPO])
        actual = run_logged("commit", ["git", "rev-parse", "HEAD"], cwd=REPO).stdout.strip()
        report["commit"] = actual
        if actual.lower() != EXPECTED_COMMIT.lower():
            raise RuntimeError("SHA không khớp; dừng để tránh kiểm thử nhầm source")
        run_logged("pin_commit", ["git", "checkout", "--detach", actual], cwd=REPO)
        run_logged("source_status", ["git", "status", "--porcelain=v1"], cwd=REPO)
        run_logged("source_diff_check", ["git", "diff", "--check"], cwd=REPO)
        manifest = {}
        tracked = run_logged("tracked_files", ["git", "ls-files"], cwd=REPO).stdout.splitlines()
        for relative in tracked:
            file = REPO / relative
            if file.is_file():
                manifest[relative] = hashlib.sha256(file.read_bytes()).hexdigest()
    (EVIDENCE / "source_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    required_sources = ("include/agent/deadline_confirmation.h",
                        "src/agent/deadline_confirmation.cpp",
                        "tests/deadline_extractor/deadline_confirmation_test.cpp")
    if any(relative not in manifest for relative in required_sources):
        raise RuntimeError("Source chưa có đủ implementation TASK-021; đóng gói lại hoặc kiểm tra commit")
    cmake_text = (REPO / "tests/deadline_extractor/CMakeLists.txt").read_text(encoding="utf-8")
    if "add_test(NAME task21_unit " not in cmake_text:
        raise RuntimeError("Source chưa đăng ký task21_unit")
    report["statuses"]["source"] = "PASS"
except Exception as error:
    report["statuses"]["source"] = "BLOCKED"
    report["error"] = str(error)
    raise
finally:
    archive()


## Build và CTest TASK-017/TASK-021

Chạy các lệnh `cmake -S tests/deadline_extractor -B build/task21`,
`cmake --build build/task21 --parallel`, `ctest --test-dir build/task21 --output-on-failure`
trong source đã pin; giới hạn parallel=2 để giảm áp lực RAM.


In [ ]:
BUILD = REPO / "build/task21"
try:
    run_logged("configure_task21", PREFIX + [CMAKE_BIN, "-S", "tests/deadline_extractor", "-B", "build/task21",
               ] + CMAKE_OPTIONS, cwd=REPO)
    run_logged("build_task21", PREFIX + [CMAKE_BIN, "--build", "build/task21", "--parallel", "2"], cwd=REPO)
    report["statuses"]["build"] = "PASS"
    result = run_logged("ctest_task21", PREFIX + [CTEST_BIN, "--test-dir", "build/task21", "--output-on-failure",
                        "--no-tests=error", "-T", "Test"], cwd=REPO, check=False)
    statuses = ctest_statuses(BUILD)
    for name in ("task17_unit", "task21_unit", "task17_http_integration", "task17_live_criteria"):
        report["statuses"][name] = statuses.get(name, "NOT RUN")
    if result.returncode or any(report["statuses"][name] != "PASS" for name in
                               ("task17_unit", "task21_unit", "task17_http_integration", "task17_live_criteria")):
        raise RuntimeError("CTest thất bại hoặc thiếu kết quả; xem XML/JSON/log")
except Exception as error:
    if report["statuses"]["build"] != "PASS":
        report["statuses"]["build"] = "FAIL"
    report["error"] = str(error)
    raise
finally:
    copy_test_artifacts(BUILD, "task21")
    archive()


In [ ]:
if RUN_ROOT_REGRESSION:
    ROOT_BUILD = REPO / "build/regression"
    try:
        run_logged("configure_root", PREFIX + [CMAKE_BIN, "-S", ".", "-B", "build/regression",
                   "-DBUILD_TESTING=ON"] + CMAKE_OPTIONS, cwd=REPO)
        run_logged("build_root", PREFIX + [CMAKE_BIN, "--build", "build/regression", "--parallel", "2"], cwd=REPO)
        result = run_logged("ctest_root", PREFIX + [CTEST_BIN, "--test-dir", "build/regression",
                            "--output-on-failure", "--no-tests=error", "-T", "Test"], cwd=REPO, check=False)
        root_statuses = ctest_statuses(ROOT_BUILD)
        report["root_tests"] = root_statuses
        required = ("agent_contract", "scheduler", "trajectory_ics", "memory_tool", "ollama_client", "ollama_http")
        if result.returncode or not root_statuses or any(value != "PASS" for value in root_statuses.values()) or any(name not in root_statuses for name in required):
            raise RuntimeError("Regression root có test thiếu/FAIL/skipped; không sửa shared source để né lỗi")
        report["statuses"]["root_regression"] = "PASS"
    except Exception as error:
        report["statuses"]["root_regression"] = "FAIL"
        report["root_error"] = str(error)
        print("Root regression FAIL:", error)
    finally:
        copy_test_artifacts(ROOT_BUILD, "regression")
        archive()


## Live tùy chọn — Ollama có sẵn trên GPU Kaggle

Khởi động Ollama/model bằng notebook TASK-013 hiện có trên Kaggle; không cần ngrok.
Đặt `RUN_LIVE=True`, `LIVE_MODEL` đúng tên/version. Cell dưới chỉ gọi model đã có,
không tải model. Probe C++ vẫn dùng `OllamaClient`; bộ tiêu chí TASK-017 đọc JSON/raw
output để chấm text đủ năm, thiếu năm, context 2026 và ảnh TKB. HTTP 200 không đủ để PASS.


In [ ]:
if RUN_LIVE:
    try:
        if not LIVE_MODEL:
            raise ValueError("Cần LIVE_MODEL tường minh")
        run_logged("gpu_live", ["nvidia-smi"])
        # Chỉ đọc metadata; extraction vẫn đi qua OllamaClient C++ hiện có.
        with urllib.request.urlopen(OLLAMA_URL + "/api/version", timeout=30) as response:
            report["ollama_version"] = json.load(response)
        show_request = urllib.request.Request(OLLAMA_URL + "/api/show",
            data=json.dumps({"model": LIVE_MODEL}).encode(), headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(show_request, timeout=30) as response:
            metadata = json.load(response)
        report["model"] = LIVE_MODEL
        with urllib.request.urlopen(OLLAMA_URL + "/api/tags", timeout=30) as response:
            tags = json.load(response)
        selected = next((model for model in tags.get("models", [])
                         if model.get("name") in (LIVE_MODEL, LIVE_MODEL + ":latest")
                         or model.get("model") in (LIVE_MODEL, LIVE_MODEL + ":latest")), None)
        if not selected or not selected.get("digest"):
            raise RuntimeError("Chưa xác định được digest của model đang dùng")
        report["model_digest"] = selected["digest"]
        report["model_modified_at"] = selected.get("modified_at")
        report["model_capabilities"] = metadata.get("capabilities", [])
        report["model_details"] = metadata.get("details", {})
        if not {"completion", "vision", "tools"}.issubset(set(metadata.get("capabilities", []))):
            raise RuntimeError("Model chưa xác nhận đủ completion/vision/tools")
        import importlib.util
        spec = importlib.util.spec_from_file_location("task17_checks", REPO / "tests/deadline_extractor/live_result_checks.py")
        checks = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(checks)
        fixtures = REPO / "tests/fixtures/deadline_extractor"
        expected = json.loads((fixtures / "live_expected.json").read_text(encoding="utf-8"))["items"]
        repository_expected = json.loads((fixtures / "input_expected.json").read_text(encoding="utf-8"))["items"]
        vision_expected = json.loads((REPO / "assets/task013/timetable_expected.json").read_text(encoding="utf-8"))
        live_cases = [
            ("text_complete", "--text", fixtures / "live_syllabus.txt", None),
            ("text_missing_year", "--text", REPO / "assets/input.txt", None),
            ("text_context_2026", "--text", REPO / "assets/input.txt", "2026"),
            ("vision_timetable", "--image", REPO / "assets/timetable.png", None)]
        live_results = []
        for name, mode, source, year in live_cases:
            result = run_logged(name, PREFIX + [BUILD / "deadline_extractor_probe", mode, source, OLLAMA_URL,
                                LIVE_MODEL] + ([year] if year else []), check=False, timeout=1200)
            try:
                if result.returncode:
                    raise RuntimeError("Probe exit " + str(result.returncode))
                data = json.loads(result.stdout)
                (EVIDENCE / (name + ".json")).write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
                (EVIDENCE / (name + ".raw.txt")).write_text(data.get("raw_model_content", ""), encoding="utf-8")
                if name == "text_complete":
                    checks.check_complete_text(data, source.read_text(encoding="utf-8"), expected)
                elif mode == "--text":
                    checks.check_repository_text(data, source.read_text(encoding="utf-8"), repository_expected,
                                                 reference_year=int(year) if year else None)
                else:
                    checks.check_vision_timetable(data, vision_expected)
                live_results.append({"name": name, "status": "PASS"})
            except Exception as error:
                live_results.append({"name": name, "status": "FAIL", "reason": str(error)})
        report["live_cases"] = live_results
        report["statuses"]["live"] = "PASS" if all(case["status"] == "PASS" for case in live_results) else "FAIL"
    except Exception as error:
        report["statuses"]["live"] = "BLOCKED"
        report["live_error"] = str(error)
    finally:
        archive()
else:
    print("Live: NOT RUN; không suy diễn PASS từ unit/HTTP mock.")
archive()
print(json.dumps(report["statuses"], ensure_ascii=False, indent=2))
print("Tải evidence ZIP:", ZIP)

print("Overall:", report["overall_status"])
